# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Strategy:** standard Google Colab notebook + GPU runtime (not Colab CLI).

Preserves: Qwen3-8B, `src/models` abstraction, fingerprinting, checkpoint/resume design.

**Before running:** Runtime → Change runtime type → **GPU**.

## 1. Mount Google Drive and find V2

1. Put your `V2/` folder on Google Drive (anywhere under My Drive).
2. Run the next cell — approve the mount prompt.
3. The cell mounts Drive and searches automatically for `V2` (`config/experiment.yaml` + `scripts/smoke_generate.py`).

In [6]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")


def is_v2(path: Path) -> bool:
    return (path / "config" / "experiment.yaml").is_file() and (
        path / "scripts" / "smoke_generate.py"
    ).is_file()


def find_v2_on_drive(max_depth: int = 8) -> Path | None:
    """Search My Drive for the V2 project root."""
    roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    found: list[Path] = []

    for root in roots:
        if not root.is_dir():
            continue
        stack: list[tuple[Path, int]] = [(root, 0)]
        while stack:
            current, depth = stack.pop()
            if is_v2(current):
                found.append(current.resolve())
            if depth >= max_depth:
                continue
            try:
                for child in current.iterdir():
                    if child.is_dir() and not child.name.startswith("."):
                        stack.append((child, depth + 1))
            except OSError:
                pass

    if not found:
        return None
    # Prefer a folder literally named V2, then shortest path
    found.sort(key=lambda p: (p.name != "V2", len(p.parts), str(p)))
    return found[0]


V2_ROOT = find_v2_on_drive()
if V2_ROOT is None:
    raise FileNotFoundError(
        "V2 not found on Google Drive. Copy the V2 folder to My Drive and re-run."
    )

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print("V2_ROOT:", V2_ROOT)

MANUAL_V2_ROOT set but invalid: /content/drive/MyDrive/.../V2
Mounting Google Drive …


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b0-3nfic6w3w75d8?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_lo

## 2. Install dependencies (Colab GPU)

In [ ]:
!pip -q install -r requirements.txt
# Preferred primary backend on Colab: CUDA llama-cpp (adjust CUDA wheel if needed)
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
# Fallback path uses transformers + bitsandbytes (already common on Colab):
# !pip -q install transformers accelerate bitsandbytes

## 3. Fingerprint + one generation smoke

Primary: `--backend llama_cpp`. Fallback: `--backend transformers`.

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers

## 4. Confirm artefacts

Expected files:
- `results/config/phase7_runtime_fingerprint.json`
- `results/config/phase7_smoke_test.json` (validation evidence: PASS/FAIL + actual output)
- `project_record/evidence/phase7_validation.md` (update Colab row after run)

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint exists:', fp.is_file())
print('smoke_test exists:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('backend:', (data.get('extra') or {}).get('backend_used'))
    print('actual:', repr(data.get('actual')))
    print('error:', data.get('error'))